# 06.5 - Regularization

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Regularization techniques prevent a model from memorizing training data by adding constraints or noise during training. The goal is better generalization to unseen data.

## 2. Why Does This Matter?

Deep networks have millions of parameters; without regularization they memorize training data and fail on test data. Overfitting is the default failure mode of deep learning.

## 3. Prerequisites

- Unit 06.4 (PyTorch MLP), basic probability

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Apply L1, L2, dropout, batch norm, early stopping
- Diagnose overfitting and choose appropriate regularization
- Understand the interaction between regularization and capacity

## 5. Mental Model

Regularization is a teacher who doesn't let students memorize the textbook. Adding noise, restricting capacity, or forcing early stopping makes the model learn general principles instead of specific examples.

- L2: `+ λ/2·||w||²` (via `weight_decay`), keeps weights small
- Dropout: randomly zero neurons during training
- Early stopping: halt when validation stops improving


## 6. Backend + Overfitting Setup

Create a small, noisy dataset where a large MLP will overfit, then compare unregularized vs regularized.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

np_X, np_y = make_blobs(n_samples=400, centers=2, n_features=2, cluster_std=2.5, random_state=42)
scaler = StandardScaler().fit(np_X)
np_X = scaler.transform(np_X)
Xtr, Xte, ytr, yte = train_test_split(np_X, np_y, test_size=0.3, random_state=42)
Xtr = torch.tensor(Xtr, dtype=torch.float32); ytr = torch.tensor(ytr, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32); yte = torch.tensor(yte, dtype=torch.float32)
print("Data ready. Noisy blobs -> large model will overfit.")


## 7. Define a Wide MLP (binary)


In [ ]:
def make_mlp(wide=True, dropout=0.0, batchnorm=False):
    layers = []
    h = 256 if wide else 8
    layers.append(nn.Linear(2, h))
    if batchnorm: layers.append(nn.BatchNorm1d(h))
    layers.append(nn.ReLU())
    if dropout: layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(h, h))
    if batchnorm: layers.append(nn.BatchNorm1d(h))
    layers.append(nn.ReLU())
    if dropout: layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(h, 1))
    layers.append(nn.Sigmoid())
    return nn.Sequential(*layers)

def evaluate(model, epochs=300, lr=0.01, weight_decay=0.0):
    crit = nn.BCELoss()
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = crit(model(Xtr), ytr.unsqueeze(1))
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        tr = accuracy_score(ytr.numpy(), (model(Xtr) > 0.5).float().numpy())
        te = accuracy_score(yte.numpy(), (model(Xte) > 0.5).float().numpy())
    return tr, te

print("Model factory ready.")


## 8. Overfitting: Wide Model WITHOUT Regularization

A wide model on a small noisy dataset memorizes noise — train accuracy far above test.


In [ ]:
# L2 via weight_decay alone
tr_plain, te_plain = evaluate(make_mlp(wide=True), weight_decay=0.0)
tr_l2,    te_l2    = evaluate(make_mlp(wide=True), weight_decay=0.1)
tr_drop,  te_drop  = evaluate(make_mlp(wide=True, dropout=0.5))

print(f"No reg:      train={tr_plain:.3f} test={te_plain:.3f}  (overfitting gap={tr_plain-te_plain:.3f})")
print(f"L2 wd=0.1:   train={tr_l2:.3f}    test={te_l2:.3f}    (gap={tr_l2-te_l2:.3f})")
print(f"Dropout .5:  train={tr_drop:.3f}  test={te_drop:.3f}  (gap={tr_drop-te_drop:.3f})")
print("\nRegularization reduces the train-test gap (less overfitting).")


## 9. Effect of L2 on Weight Magnitudes

L2 penalty keeps weights small. Inspect the distribution of weight norms.


In [ ]:
m_plain = make_mlp(wide=True)
m_l2 = make_mlp(wide=True)
evaluate(m_plain, epochs=300)
opt = optim.Adam(m_l2.parameters(), lr=0.01, weight_decay=0.2)
crit = nn.BCELoss()
for _ in range(300):
    m_l2.train(); opt.zero_grad(); crit(m_l2(Xtr), ytr.unsqueeze(1)).backward(); opt.step()

def max_weight(m):
    return max(p.abs().max().item() for p in m.parameters() if p.dim() >= 1)
print(f"Max weight (no reg):   {max_weight(m_plain):.3f}")
print(f"Max weight (L2 wd=.2): {max_weight(m_l2):.3f}")
print("\nL2 weight decay shrinks the largest weights.")


## 10. Early Stopping

Monitor validation loss and keep the best state; halt training once patience is exhausted.


In [ ]:
def train_early(model, epochs=400, patience=30):
    crit = nn.BCELoss()
    opt = optim.Adam(model.parameters(), lr=0.01)
    best_val, best_state, counter = float('inf'), None, 0
    # Use a held-out slice of training as validation
    Xv, yv = Xtr[:100], ytr[:100]
    Xt, yt = Xtr[100:], ytr[100:]
    stop_epoch = epochs
    for epoch in range(epochs):
        model.train(); opt.zero_grad(); crit(model(Xt), yt.unsqueeze(1)).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xv), yv.unsqueeze(1)).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                stop_epoch = epoch
                break
    model.load_state_dict(best_state)
    return stop_epoch

m = make_mlp(wide=True)
stopped = train_early(m)
with torch.no_grad():
    te = accuracy_score(yte.numpy(), (m(Xte) > 0.5).float().numpy())
print(f"Early stopping halted at epoch {stopped} (patience=30)")
print(f"Test accuracy using best state: {te:.3f}")
print("\nEarly stopping is a simple safety net that keeps the best model.")


## 11. Decision Guidance

| Technique | Best For | Effort | Risk |
|---|---|---|---|
| L2 weight decay | General-purpose, always use | Low | None |
| Dropout | Large nets, high overfitting risk | Low | Can slow convergence |
| Batch norm | Deep nets, stability | Medium | Adds inference complexity |
| Early stopping | Universal safety net | Low | May stop early |
| Data augmentation | Vision/NLP, limited data | Medium | Domain knowledge needed |

## 12. Common Mistakes

- Using dropout during evaluation (use `model.eval()`).
- Forgetting weight decay when switching optimizers.
- Dropout too high (can't learn) or too low (no effect).
- Batch norm on tiny batch sizes (noisy statistics).

## 13. Debugging / Troubleshooting

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Low train acc with dropout | Dropout too high | Reduce p | Use 0.1-0.3 |
| Val loss unstable | BN small batch | Check batch size | Bigger batch or LayerNorm |
| Same with/without reg | Underfitting | Check train vs val | Increase capacity first |
| Early stop too early | Patience low | Plot val loss | Increase patience |

## 14. When NOT to Use

- Heavy regularization before confirming overfitting (address capacity first).

## 15. Challenge

Reduce model width dramatically and confirm the gap narrows — capacity directly drives overfitting.


In [ ]:
# Challenge: capacity vs overfitting
tr_narrow, te_narrow = evaluate(make_mlp(wide=False), weight_decay=0.0)
print(f"Narrow model: train={tr_narrow:.3f} test={te_narrow:.3f} (gap={tr_narrow-te_narrow:.3f})")
print("Smaller capacity overfits less, but may underfit if too small.")


## 16. Closed-Book Recall

Without looking back:

1. Why does L2 promote small (not zero) weights while L1 promotes sparsity?
2. Why must dropout be disabled during evaluation?
3. How does batch norm help both training speed and regularization?
4. When is early stopping insufficient?

## 17. Teach-Back Questions

Explain to another person:

- How to detect overfitting from train/test curves.
- The role of each regularization technique.

## 18. Summary

You compared unregularized vs L2 vs dropout, verified L2 shrinks weights, and implemented early stopping.

## 19. Further Experiment

- Combine dropout + weight decay + early stopping and measure the combined effect.
- Compare batch norm vs no batch norm.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
